In [1]:
from FuncoesProbNet import *

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

# ==========================================================
# PARÂMETROS
# ==========================================================

torch.set_default_dtype(torch.float64)

R0 = torch.tensor(0.15)
r0 = 0.15

delta_r = 0.0025
delta_t = 32/252

n_steps = 8

IDI0 = 100000.0

# ==========================================================
# TARGET YIELD
# ==========================================================

target_yield = torch.tensor([0.15000000, 0.15084385, 0.15143079, 0.15133712, 0.15085813, 0.15017317, 0.14937110, 0.14849885], dtype=torch.float64)

# ==========================================================
# SUPERFÍCIES
# ==========================================================

strike_surface = {
    1: torch.tensor([101713.99999955237, 101819.7619047619, 101935.71428616189]),
    2: torch.tensor([103636.09607431028, 103743.57142857142, 103861.42963997542]),
    3: torch.tensor([105577.2228032737, 105686.42857142857, 105806.21148244056]),
    4: torch.tensor([107499.31887803161, 107610.23809523809, 107731.92683625409]),
    5: torch.tensor([109373.8383172757, 109486.42857142857, 109609.97596843856]),
    6: torch.tensor([111229.32710231429, 111343.57142857143, 111468.95861197144]),
    7: torch.tensor([113065.78523314734, 113181.66666666667, 113308.87476685268]),
    8: torch.tensor([114913.6617565037, 115031.19047619049, 115160.23081492489]),
}

market_surface = {
    1: torch.tensor([187.19626212149666, 83.41121495327104, 1.0]),
    2: torch.tensor([223.14297398840048, 119.66982648345268, 11.094096092944998]),
    3: torch.tensor([288.1369872928479, 184.9981633559357, 72.86712473915073]),
    4: torch.tensor([380.56736344502673, 277.7787169500917, 165.0785247727232]),
    5: torch.tensor([521.3860753923154, 418.9820398673316, 306.6132295587394]),
    6: torch.tensor([683.6485518416397, 581.6401161790204, 469.6822564901476]),
    7: torch.tensor([869.6713638968536, 768.0724180804291, 656.5428271382154]),
    8: torch.tensor([1057.3700133948903, 956.175773762969, 845.0698330629193]),
}

# ==========================================================
# REDE
# ==========================================================

class ProbNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(2,64),
            nn.Tanh(),

            nn.Linear(64,64),
            nn.Tanh(),

            nn.Linear(64,3)
        )

    def forward(self,i,j):

        x = torch.tensor([i,j],dtype=torch.float64)

        logits = self.net(x)

        probs = torch.softmax(logits,dim=0)

        return probs

model = ProbNet()
optimizer = optim.Adam(model.parameters(),lr=1e-3)

# ==========================================================
# LOSS
# ==========================================================

def loss_function(current_lambda_options):

    probs = generate_probs(n_steps, model)

    # ==========================
    # YIELD LOSS
    # ==========================

    _,y_model = build_tree_and_yield_torch(
        probs, n_steps, R0, delta_r, delta_t, torch
    )

    loss_yield = torch.mean((y_model-target_yield)**2)

    # ==========================
    # OPTION SURFACE LOSS
    # ==========================

    surface_model = price_all_options_surface(
        probs, n_steps, strike_surface, torch, IDI0, R0, delta_r, delta_t
    )

    loss_options = torch.tensor(0.0)

    weights_strike = torch.tensor([1.0, 3.0, 1.0])

    for step in range(1,n_steps+1):

        C_market = market_surface[step]
        C_model = surface_model[step]

        T = step*delta_t
        weight_T = 1.0/T

        scale = torch.maximum(C_market, torch.tensor(1.0))

        rel_error = (C_model - C_market)**2 / scale

        weighted_error = weights_strike * rel_error

        loss_options += weight_T * torch.mean(weighted_error)

    loss_options = loss_options/n_steps

    loss = loss_yield + current_lambda_options*loss_options

    return loss,y_model,surface_model

# ==========================================================
# TREINO
# ==========================================================

loss_history = []

for epoch in range(1):

    current_lambda_options = 0.001

    optimizer.zero_grad()

    loss,y,surface = loss_function(current_lambda_options)

    loss.backward()

    optimizer.step()

    loss_history.append(loss.item())

    if epoch%100==0:

        print("epoch:",epoch)
        print("loss:",loss.item())
        print("yield:",y.detach())
        print("surface:",surface)
        print()

# ==========================================================
# RESULTADOS
# ==========================================================

probs = generate_probs(n_steps, model)

_,y_final = build_tree_and_yield_torch(
    probs, n_steps, R0, delta_r, delta_t, torch
)

surface_final = price_all_options_surface(
    probs, n_steps, strike_surface, torch, IDI0, R0, delta_r, delta_t
)

# ==========================================================
# CHECK
# ==========================================================

print("\nCHECK STEP 2:")
print("strikes:", strike_surface[2])
print("model  :", surface_final[2])
print("market :", market_surface[2])

# ==========================================================
# PLOTS
# ==========================================================

plot_loss(loss_history, plt)

plot_yield_curve(y_final, target_yield, np, plt)
plot_yield_curve(y_final, target_yield, np, plt, y_lim=(0.13, 0.17))

plot_option_prices_surface(surface_final, market_surface, strike_surface, plt)

plot_trinomial_tree2(probs, n_steps, r0, delta_r, np, plt)

NameError: name 'n_steps' is not defined